# MT3 on Apple Silicon: official JAX inference without the T5X installation stack

The original plan was to correct the unofficial PyTorch port. We instead use the **official MT3 network and checkpoint directly**, plus the official spectrogram/event functions and standalone T5X beam search. This avoids the port's positional-encoding and EOS bugs entirely.

No `pip install t5x`, `tensorflow-cpu`, SeqIO, or T5 installation is required. The small runtime still uses the existing Mac-compatible TensorFlow for official spectrogram computation. The source revisions, generated inference-package hashes and checkpoint tensor hashes are recorded.

**Run All performs setup and bounded checks only.** It does not run a dataset benchmark. The setup downloads dependencies/source/approximately 175 MiB of inference weights if absent; later runs reuse them. Full-audio transcription is opt-in below. This is an inference wrapper around upstream code, not a new model or retraining experiment.


In [ ]:
from pathlib import Path
import sys, subprocess, json
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / 'app.py').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from benchmarks.modules.note.experiments.MT3Direct import MT3Direct, ASSETS, MT3_REV, T5X_REV
HOST = ROOT / 'at-venv/bin/python'
PYTHON = ASSETS / 'venv/bin/python'
SCRIPT = ROOT / 'benchmarks/modules/note/experiments/MT3Direct.py'
REPORTS = ROOT / 'benchmarks/results/note/mt3-direct'
REPORTS.mkdir(parents=True, exist_ok=True)
print({'official_mt3': MT3_REV, 'official_t5x_decoder': T5X_REV, 'assets': str(ASSETS)})

def run(action, *arguments, host=False):
    subprocess.run([str(HOST if host else PYTHON), str(SCRIPT), action, *map(str, arguments)], check=True)


## Reproducible local setup

A private virtual environment under the ignored benchmark assets directory reads the existing `at-venv` packages through a `.pth` file. Additional pinned dependencies are installed only in that environment; the host environment is not modified. Consequently, this is a development overlay rather than a fully standalone environment: benchmark provenance records its resolved packages, and changes to the host can invalidate caches.

Only the network, spectral/event functions, and beam-search modules are imported. Training/dataset package initializers are replaced with empty ones. The vocabulary helpers and run-length event decoder are selected from official source, preserving their function bodies. No model weights or network equations are patched.


In [ ]:
run('setup', host=True)
if not (ASSETS / 'checkpoint/params.npz').exists():
    run('download', '--output', REPORTS / 'download.json')
else:
    print('Reusing downloaded official inference tensors; runtime verifies their hashes.')


## Bounded validation

The check loads all 189 official tensors and verifies their recorded hashes. It compares cached decoding to full-prefix decoding for three tokens, checks a hand-constructed note event stream with and without EOS, and runs one 0.25-second synthetic signal through official spectrogram computation and beam search, capped at 32 tokens. This cap is only for the smoke check; the transcription runner uses the official 1024-token limit.

A pass validates the local inference plumbing. It is not a corpus quality result or an independent end-to-end parity comparison against an installed Colab/T5X wrapper. The latter has not been performed. Network, weights and beam search come from upstream rather than the unverified port.


In [ ]:
run('smoke', '--output', REPORTS / 'smoke.json')
smoke = json.loads((REPORTS / 'smoke.json').read_text())
display(smoke)
assert max(smoke['cache_logit_max_abs_errors']) < 1e-3
assert smoke['event_decode_passed']


## Optional transcription of a short clip

Set `AUDIO_PATH` and `RUN_CLIP=True` to transcribe an actual file. The entire selected file runs, so choose a short clip first. No score, annotated pitch range, or reference-note boundaries are supplied. All predicted instruments and simultaneous notes are retained. A subprocess keeps the JAX runtime isolated from the notebook kernel.


In [ ]:
RUN_CLIP = False
AUDIO_PATH = None
if RUN_CLIP:
    if AUDIO_PATH is None:
        raise ValueError('Choose a short audio file first')
    run('predict', '--audio', Path(AUDIO_PATH), '--output', REPORTS / 'clip.json')
    display(json.loads((REPORTS / 'clip.json').read_text()))
else:
    print('Full-file transcription is disabled; no benchmark started.')


## Use in note detection comparisons

The registered method is **`mt3`**, backed by official JAX MT3. It is included in the default methods in `note.ipynb`. After setup, restart that notebook's kernel and Run All. Its setup cell previews the automatically discovered cached and pending jobs; compatible existing results are reused without pinned run paths or method-specific switches.

MT3 runs one recording worker at a time, in its own CPU subprocess. Source, checkpoint and runtime dependency fingerprints participate in result reuse. CPU accounting includes the reaped inference subprocess; compilation/setup is included and is not steady-state latency. Summary/significance cells using `CONFIG.methods` include MT3 automatically.

Before making generalization claims, audit the official checkpoint's training datasets against this suite. A preliminary comparison is not proof of held-out performance, and no numerical results have been generated here.

Sources: [official MT3](https://github.com/magenta/mt3), [T5X decoder](https://github.com/google-research/t5x/blob/main/t5x/decoding.py), [earlier port audit](../../docs/mt3-pytorch-equivalence-audit-2026-10-14.md).
